# 10 – Results: Limitaciones del Proyecto

**Proyecto:** Análisis y predicción del subempleo por insuficiencia de horas en el Perú – EPEN 2024  
**Target:** `target_subempleo_horas`  
**Objetivo:** Documentar de forma transparente las limitaciones metodológicas, de datos y éticas del modelo.


## 1. Limitaciones de los datos

### 1.1 Representatividad y sesgo de selección
- La EPEN captura únicamente a hogares en viviendas particulares; **quedan excluidos** trabajadores en situación de calle, migrantes en tránsito e instituciones colectivas.
- Existe un posible **sesgo de no-respuesta**: los hogares con mayor movilidad o informalidad extrema tienen menor probabilidad de ser encuestados.

### 1.2 Calidad de las variables de ingresos y horas
- Los ingresos y horas trabajadas están **auto-reportados**, lo que puede subestimar ingresos reales (especialmente en el sector informal).
- Las variables de ingreso (`ingtrabw`, `INGTOT`) pueden no capturar ingresos esporádicos o en especie.

### 1.3 Corte temporal
- El modelo es estático y fue entrenado con datos de la EPEN 2024. Cambios macroeconómicos (crisis, reformas laborales, estacionalidad) pueden afectar la validez de las predicciones en otros períodos.

### 1.4 Definición de la variable objetivo
- Se usa la definición operacional de **subempleo por insuficiencia de horas** basada en `P209H` (voluntad y disponibilidad de trabajar más horas). Esta definición captura la dimensión subjetiva del fenómeno y puede diferir de definiciones basadas únicamente en horas efectivas trabajadas.
- Trabajadores que ya tienen un segundo empleo o que trabajan exactamente las horas que desean no son capturados por esta definición.


## 2. Limitaciones metodológicas

### 2.1 Generalización temporal
- El modelo **no debe extrapolarse** a períodos con condiciones económicas muy distintas al trimestre de entrenamiento.
- Se recomienda **reentrenar periódicamente** (al menos cada trimestre) con nuevas olas de la EPEN.

### 2.2 Desbalance de clases residual
- La estrategia `class_weight='balanced'` y las técnicas de remuestreo (RandomOverSampler, RandomUnderSampler) mitigan el desbalance, pero no lo eliminan.
- El modelo ganador (LR balanceada) alcanza un Recall de ~0.57 para la clase positiva, lo que implica que ~43 % de los subempleados por horas no son detectados correctamente.

### 2.3 Causalidad vs. correlación
- El modelo identifica **correlaciones** entre variables laborales y el subempleo por horas, pero **no establece relaciones causales**. Las variables predictoras están correlacionadas con el target, no necesariamente lo causan.

### 2.4 Capacidad predictiva moderada
- El ROC-AUC del mejor modelo es ~0.697 (sobre 0.50 del azar), lo que representa capacidad discriminativa moderada. Existe margen de mejora mediante ingeniería de características adicional, otros algoritmos (gradient boosting) o datos de más períodos.

### 2.5 Hiperparámetros y validación
- La búsqueda de hiperparámetros se realizó con GridSearchCV sobre 5-fold CV. Una búsqueda más exhaustiva (RandomizedSearchCV, Bayesian Optimization) podría mejorar resultados.
- No se realizó validación temporal (walk-forward) dado que la EPEN es transversal.


## 3. Limitaciones de interpretabilidad

- La **Regresión Logística** ofrece interpretabilidad directa mediante coeficientes, pero asume linealidad en el log-odds.
- Los coeficientes no explican predicciones individuales de forma intuitiva; para eso se recomienda incorporar **SHAP** (disponible como paso opcional en `09_evaluation/feature_importance.ipynb`).
- Variables altamente correlacionadas entre sí (ej: múltiples medidas de ingresos) pueden generar coeficientes inestables que dificultan la interpretación.


## 4. Limitaciones éticas y de equidad

- **Sesgo algorítmico:** Si históricamente ciertos grupos (trabajadores informales, mujeres, jóvenes, poblaciones indígenas) tienen mayor prevalencia de subempleo por horas, el modelo puede perpetuar esos patrones sin ofrecer una explicación causal.
- Se recomienda realizar un **análisis de equidad** (fairness analysis) evaluando métricas de desempeño por grupo demográfico (`C207` sexo, `C208` edad, `C377` etnicidad) antes de cualquier uso en política pública.
- El modelo **no debe usarse** para tomar decisiones automáticas que afecten derechos laborales o acceso a programas sociales sin supervisión humana.
- La predicción de subempleo no equivale a diagnóstico laboral. Factores contextuales, preferencias individuales y condiciones del mercado local no están capturados en el modelo.


## 5. Próximos pasos recomendados

| # | Acción | Prioridad |
|---|---|---|
| 1 | Incorporar datos longitudinales para capturar transiciones laborales | Alta |
| 2 | Integrar variables macroeconómicas (tasa de inflación, PIB sectorial) | Alta |
| 3 | Realizar análisis de equidad por sexo, edad y nivel educativo | Alta |
| 4 | Implementar validación temporal (walk-forward) | Media |
| 5 | Explorar modelos de gradient boosting (XGBoost, LightGBM) | Media |
| 6 | Desarrollar un dashboard interactivo para comunicar resultados | Baja |

In [ ]:
import pandas as pd

# Resumen estructurado de limitaciones del proyecto
limitaciones = [
    {'Categoría': 'Datos',             'Limitación': 'Sesgo de selección: excluye trabajadores sin hogar fijo'},
    {'Categoría': 'Datos',             'Limitación': 'Ingresos y horas auto-reportados (posible subestimación)'},
    {'Categoría': 'Datos',             'Limitación': 'Corte temporal único (EPEN 2024); sin validación longitudinal'},
    {'Categoría': 'Datos',             'Limitación': 'Definición subjetiva del target (voluntad y disponibilidad)'},
    {'Categoría': 'Metodología',       'Limitación': 'Modelo correlacional, no causal'},
    {'Categoría': 'Metodología',       'Limitación': 'Recall clase 1 ~0.57: ~43% de subempleados no detectados'},
    {'Categoría': 'Metodología',       'Limitación': 'ROC-AUC ~0.697: capacidad discriminativa moderada'},
    {'Categoría': 'Metodología',       'Limitación': 'GridSearchCV básico; no se exploró Bayesian Optimization'},
    {'Categoría': 'Interpretabilidad', 'Limitación': 'Coeficientes LR asumen log-odds lineal'},
    {'Categoría': 'Ética',             'Limitación': 'Posible perpetuación de sesgos demográficos históricos'},
    {'Categoría': 'Ética',             'Limitación': 'No reemplaza análisis humano en política laboral'},
]

lim_df = pd.DataFrame(limitaciones)
print(lim_df.to_string(index=False))
